In [0]:
%sql
USE CATALOG f1_catalog;
CREATE SCHEMA IF NOT EXISTS f1_catalog.ml_process;

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.ml_process.driver_race_features AS
SELECT r.race_id,r.race_year,r.round,d.driver_id,d.full_name AS driver_name,c.name AS circuit_name,
res.grid AS grid_position,res.position AS final_position,res.points,
CASE WHEN res.position=1 THEN 1 ELSE 0 END AS is_winner,q.position AS quali_position,
AVG(res.position) OVER (PARTITION BY res.driver_id ORDER BY r.race_id ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS avg_position_last5,
SUM(res.points) OVER (PARTITION BY res.driver_id ORDER BY r.race_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS total_points_before_race
FROM f1_catalog.silver.results res JOIN f1_catalog.silver.races r ON res.race_id = r.race_id
JOIN f1_catalog.silver.drivers d ON res.driver_id=d.driver_id
LEFT JOIN f1_catalog.silver.qualifying q ON res.race_id = q.race_id AND res.driver_id = q.driver_id
JOIN f1_catalog.silver.circuits c ON r.circuit_id = c.circuit_id;

select * from f1_catalog.ml_process.driver_race_features

In [0]:
ml_df=spark.read.table('f1_catalog.ml_process.driver_race_features')


In [0]:
ml_df=ml_df.dropna()
display(ml_df)

In [0]:
ml_df=ml_df.dropDuplicates()
display(ml_df)

In [0]:
from pyspark.ml.feature import *

feature_col=['grid_position','quali_position','avg_position_last5','total_points_before_race']

assemble=VectorAssembler(inputCols=feature_col,outputCol='features')

ml_df=assemble.transform(ml_df).select('features','is_winner')


train_df,test_df=ml_df.randomSplit([0.8,0.2],seed=42)


In [0]:
from pyspark.ml.classification import RandomForestClassifier
random_forest=RandomForestClassifier(labelCol="is_winner", featuresCol="features")
model=random_forest.fit(train_df)

In [0]:
predictions=model.transform(test_df)
display(predictions)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator=BinaryClassificationEvaluator(labelCol="is_winner")

accuracy=evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

In [0]:
predictions=model.transform(test_df)

In [0]:
display(predictions)

In [0]:
predictions.filter("prediction = 1").count()

### testing of xgboost model

In [0]:
display(ml_df)

In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
pdf=ml_df.toPandas()

In [0]:
y=pdf['is_winner']
features=['driver_id','grid_position','quali_position','avg_position_last5','total_points_before_race']
x=pdf[features]


In [0]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

model=XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=5,
scale_pos_weight=10,random_state=42)

model.fit(x_train,y_train)
y_pred=model.predict(x_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


this model is not good for predicting winner so we need to tune the model

In [0]:
model=XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=5,
scale_pos_weight=20,random_state=42)

model.fit(x_train,y_train)
y_pred=model.predict(x_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [0]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

Try adding more features to check accuracy

In [0]:
y=pdf['is_winner']
features=['driver_id','grid_position','quali_position','avg_position_last5','total_points_before_race','round','race_year']
x=pdf[features]

In [0]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

model=XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=5,
scale_pos_weight=10,random_state=42)

model.fit(x_train,y_train)
y_pred=model.predict(x_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [0]:
print(classification_report(y_test, y_pred))

this moel is very good at predicting non winners,the data is imbalance

In [0]:
model = XGBClassifier(n_estimators=300,learning_rate=0.05,
max_depth=6,scale_pos_weight=25,subsample=0.8,colsample_bytree=0.8,
random_state=42)

model.fit(x_train,y_train)
y_pred=model.predict(x_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [0]:
y_prob=model.predict_proba(x_test)[:,1]
y_pred=(y_prob > 0.7).astype(int)   # increase threshold

In [0]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))